In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModel, AutoTokenizer, AutoImageProcessor
from datasets import load_dataset
from PIL import Image
import numpy as np
from tqdm import tqdm
import time
import gc
import copy

In [2]:
device = 'cuda'

In [3]:
class SimpleVLM(nn.Module):
    """Simple Vision-Language Model for image captioning."""
    def __init__(self, vision_model_name="google/vit-base-patch16-224",
                 language_model_name="HuggingFaceTB/SmolLM-135M"):
        super().__init__()

        self.vision_encoder = AutoModel.from_pretrained(vision_model_name)
        self.vision_hidden_size = self.vision_encoder.config.hidden_size  # 768 for ViT-Base

        self.language_model = AutoModel.from_pretrained(language_model_name)
        self.language_hidden_size = self.language_model.config.hidden_size  # 576 for SmolLM

        self.projector = nn.Linear(self.vision_hidden_size, self.language_hidden_size)

        self.lm_head = nn.Linear(self.language_hidden_size,
                                 self.language_model.config.vocab_size, bias=False)

    def forward(self, pixel_values, input_ids, attention_mask=None, labels=None):

        vision_outputs = self.vision_encoder(pixel_values=pixel_values)
        image_features = vision_outputs.last_hidden_state  # [batch, 197, 768]

        image_embeds = self.projector(image_features)  # [batch, 197, 576]

        text_embeds = self.language_model.embed_tokens(input_ids)  # [batch, seq_len, 576]

        combined_embeds = torch.cat([image_embeds, text_embeds], dim=1)  # [batch, 197+seq_len, 576]

        batch_size = pixel_values.shape[0]
        image_attention = torch.ones(batch_size, image_embeds.shape[1],
                                     dtype=torch.long, device=pixel_values.device)
        if attention_mask is None:
            attention_mask = torch.ones_like(input_ids)
        combined_attention = torch.cat([image_attention, attention_mask], dim=1)

        outputs = self.language_model(
            inputs_embeds=combined_embeds,
            attention_mask=combined_attention,
            output_hidden_states=True,
            return_dict=True
        )

        hidden_states = outputs.last_hidden_state
        logits = self.lm_head(hidden_states)

        loss = None
        if labels is not None:
            shift_logits = logits[:, image_embeds.shape[1]:-1, :].contiguous()
            shift_labels = labels[:, 1:].contiguous()

            loss_fct = nn.CrossEntropyLoss(ignore_index=-100)
            loss = loss_fct(shift_logits.view(-1, shift_logits.size(-1)),
                          shift_labels.view(-1))

        return {"loss": loss, "logits": logits, "hidden_states": hidden_states}

    def count_parameters(self):
        """Count total trainable parameters."""
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

    def get_model_size_mb(self):
        """Calculate model size in MB."""
        param_size = sum(p.numel() * p.element_size() for p in self.parameters())
        buffer_size = sum(b.numel() * b.element_size() for b in self.buffers())
        return (param_size + buffer_size) / 1024**2

In [4]:
baseline_model = SimpleVLM().to(device)

config.json:   0%|          | 0.00/69.7k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] ViTModel LOAD REPORT from: google/vit-base-patch16-224
Key                 | Status     | 
--------------------+------------+-
classifier.bias     | UNEXPECTED | 
classifier.weight   | UNEXPECTED | 
pooler.dense.bias   | MISSING    | 
pooler.dense.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


config.json:   0%|          | 0.00/724 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/538M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

In [5]:
print(f"Baseline Model Statistics:")
print(f"  Total parameters: {baseline_model.count_parameters() / 1e6:.1f}M")
print(f"  Model size: {baseline_model.get_model_size_mb():.1f} MB")
print(f"\nComponent breakdown:")
print(f"  Vision encoder: {sum(p.numel() for p in baseline_model.vision_encoder.parameters()) / 1e6:.1f}M")
print(f"  Language model: {sum(p.numel() for p in baseline_model.language_model.parameters()) / 1e6:.1f}M")
print(f"  Projector: {sum(p.numel() for p in baseline_model.projector.parameters()) / 1e6:.1f}M")
print(f"  LM head: {sum(p.numel() for p in baseline_model.lm_head.parameters()) / 1e6:.1f}M")

Baseline Model Statistics:
  Total parameters: 249.7M
  Model size: 695.8 MB

Component breakdown:
  Vision encoder: 86.4M
  Language model: 134.5M
  Projector: 0.4M
  LM head: 28.3M


In [6]:
tokenizer = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM-135M")
tokenizer.pad_token = tokenizer.eos_token
image_processor = AutoImageProcessor.from_pretrained("google/vit-base-patch16-224")

tokenizer_config.json:   0%|          | 0.00/3.69k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/801k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/831 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.10M [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/160 [00:00<?, ?B/s]

# Dataset

In [7]:
dataset = load_dataset("jxie/flickr8k", split="train")
dataset = dataset.shuffle(seed=42).select(range(500))  # Small subset for speed
print(f"Loaded {len(dataset)} examples for evaluation")

README.md:   0%|          | 0.00/687 [00:00<?, ?B/s]

data/train-00000-of-00002-2f8f6bfa852eac(…):   0%|          | 0.00/414M [00:00<?, ?B/s]

data/train-00001-of-00002-2173151d8cd6c7(…):   0%|          | 0.00/414M [00:00<?, ?B/s]

data/validation-00000-of-00001-7025a2b59(…):   0%|          | 0.00/138M [00:00<?, ?B/s]

data/test-00000-of-00001-42a2661d12c73e4(…):   0%|          | 0.00/137M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/6000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Loaded 500 examples for evaluation


In [8]:
class CaptionDataset(Dataset):
    def __init__(self, dataset, image_processor, tokenizer, max_length=64):
        self.dataset = dataset
        self.image_processor = image_processor
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        item = self.dataset[idx]
        image = item['image'].convert('RGB')
        # flickr8k uses caption_0, caption_1, etc. - use first caption
        caption = item['caption_0']

        # Process image
        pixel_values = self.image_processor(image, return_tensors="pt")["pixel_values"][0]

        # Process caption
        prompt = "Caption: "
        full_text = f"{prompt}{caption}{self.tokenizer.eos_token}"

        encoding = self.tokenizer(
            full_text,
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )

        input_ids = encoding["input_ids"][0]
        attention_mask = encoding["attention_mask"][0]

        # Create labels (mask prompt tokens)
        labels = input_ids.clone()
        prompt_len = len(self.tokenizer(prompt, add_special_tokens=False)["input_ids"])
        labels[:prompt_len] = -100
        labels[attention_mask == 0] = -100

        return {
            "pixel_values": pixel_values,
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels,
            "caption": caption
        }

eval_dataset = CaptionDataset(dataset, image_processor, tokenizer)
eval_loader = DataLoader(eval_dataset, batch_size=8, shuffle=False)
print(f"Created DataLoader with {len(eval_loader)} batches")

Created DataLoader with 63 batches


In [9]:
def evaluate_model(model, dataloader, num_batches=50):
    """Evaluate model performance."""
    model.eval()
    total_loss = 0
    total_time = 0
    num_samples = 0

    with torch.no_grad():
        for i, batch in enumerate(tqdm(dataloader, total=num_batches, desc="Evaluating")):
            if i >= num_batches:
                break

            pixel_values = batch["pixel_values"].to(device)
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            # Measure inference time
            start_time = time.time()
            outputs = model(pixel_values, input_ids, attention_mask, labels)
            torch.cuda.synchronize() if torch.cuda.is_available() else None
            end_time = time.time()

            total_loss += outputs["loss"].item() * pixel_values.size(0)
            total_time += (end_time - start_time)
            num_samples += pixel_values.size(0)

    avg_loss = total_loss / num_samples
    avg_time_per_sample = total_time / num_samples
    throughput = num_samples / total_time

    return {
        "loss": avg_loss,
        "time_per_sample": avg_time_per_sample,
        "throughput": throughput
    }

print("Evaluating baseline model...")
baseline_model = baseline_model.float() # Cast model to float32
baseline_results = evaluate_model(baseline_model, eval_loader)
print(f"\nBaseline Results:")
print(f"  Loss: {baseline_results['loss']:.4f}")
print(f"  Time per sample: {baseline_results['time_per_sample']*1000:.2f} ms")
print(f"  Throughput: {baseline_results['throughput']:.2f} samples/sec")

Evaluating baseline model...


Evaluating: 100%|██████████| 50/50 [00:17<00:00,  2.80it/s]


Baseline Results:
  Loss: 11.1455
  Time per sample: 37.25 ms
  Throughput: 26.85 samples/sec


#Int8 Quantization

In [10]:
baseline_model_cpu = copy.deepcopy(baseline_model).cpu()
baseline_model_cpu.eval()

quantized_model = torch.quantization.quantize_dynamic(
    baseline_model_cpu,
    {nn.Linear},
    dtype=torch.qint8
)

/tmp/ipykernel_665/2744110081.py:4: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  quantized_model = torch.quantization.quantize_dynamic(


In [11]:
print(f"\nQuantized Model Statistics:")
print(f"  Model size: {quantized_model.get_model_size_mb():.1f} MB")
print(f"  Size reduction: {baseline_model.get_model_size_mb() / quantized_model.get_model_size_mb():.2f}x")
print(f"\nNote: Dynamic quantization runs on CPU only (PyTorch limitation).")


Quantized Model Statistics:
  Model size: 111.1 MB
  Size reduction: 8.57x

Note: Dynamic quantization runs on CPU only (PyTorch limitation).


In [12]:
def evaluate_model_cpu(model, dataloader, num_batches=20):
    """Evaluate model on CPU (for quantized models)."""
    model.eval()
    total_loss = 0
    total_time = 0
    num_samples = 0

    with torch.no_grad():
        for i, batch in enumerate(tqdm(dataloader, total=num_batches, desc="Evaluating (CPU)")):
            if i >= num_batches:
                break

            pixel_values = batch["pixel_values"]  # Already on CPU from dataloader
            input_ids = batch["input_ids"]
            attention_mask = batch["attention_mask"]
            labels = batch["labels"]

            start_time = time.time()
            outputs = model(pixel_values, input_ids, attention_mask, labels)
            end_time = time.time()

            total_loss += outputs["loss"].item() * pixel_values.size(0)
            total_time += (end_time - start_time)
            num_samples += pixel_values.size(0)

    avg_loss = total_loss / num_samples
    avg_time_per_sample = total_time / num_samples
    throughput = num_samples / total_time

    return {
        "loss": avg_loss,
        "time_per_sample": avg_time_per_sample,
        "throughput": throughput
    }

print("Evaluating quantized model on CPU...")
print("(Note: Quantized models run on CPU; comparing CPU vs GPU isn't apples-to-apples)")
quantized_results = evaluate_model_cpu(quantized_model, eval_loader, num_batches=10)

print(f"\nQuantized Results (CPU):")
print(f"  Loss: {quantized_results['loss']:.4f} (vs baseline {baseline_results['loss']:.4f})")
print(f"  Loss change: {(quantized_results['loss'] - baseline_results['loss']) / baseline_results['loss'] * 100:+.2f}%")
print(f"  Time per sample: {quantized_results['time_per_sample']*1000:.2f} ms (CPU)")
print(f"  Throughput: {quantized_results['throughput']:.2f} samples/sec (CPU)")
print(f"\nKey benefit: Model size reduced from {baseline_model.get_model_size_mb():.0f} MB to {quantized_model.get_model_size_mb():.0f} MB ({baseline_model.get_model_size_mb() / quantized_model.get_model_size_mb():.1f}x smaller)")


Evaluating quantized model on CPU...
(Note: Quantized models run on CPU; comparing CPU vs GPU isn't apples-to-apples)


Evaluating (CPU): 100%|██████████| 10/10 [01:36<00:00,  9.64s/it]



Quantized Results (CPU):
  Loss: 11.1852 (vs baseline 11.1455)
  Loss change: +0.36%
  Time per sample: 1197.11 ms (CPU)
  Throughput: 0.84 samples/sec (CPU)

Key benefit: Model size reduced from 952 MB to 111 MB (8.6x smaller)


# Structured Pruning

In [13]:
def compute_head_importance(model, dataloader, num_batches = 20):
  model.eval()
  num_layers = len(model.language_model.layers)
  num_heads = model.language_model.config.num_attention_heads

  head_importance = torch.zeros(num_layers, num_heads).to(device)

  with torch.no_grad():
    for i, batch in enumerate(tqdm(dataloader, total=num_batches, desc = "Computing importance")):
      if i >= num_batches:
        break

      pixel_values = batch["pixel_values"].to(device)
      input_ids = batch["input_ids"].to(device)
      attention_mask = batch["attention_mask"].to(device)

      outputs = model(pixel_values, input_ids, attention_mask)

      for layer_idx in range(num_layers):
        head_importance[layer_idx] += torch.randn(num_heads).to(device)

  head_importance /= num_batches

  return head_importance

In [14]:
def prune_attention_heads(model, head_importance, prune_ratio=0.3):
  flat_importance = head_importance.view(-1)
  num_heads_total = flat_importance.size(0)
  num_to_prune = int(num_heads_total * prune_ratio)

  _, indices_to_prune = torch.topk(flat_importance, num_to_prune, largest=False)

  print(f"Pruning {num_to_prune} out of {num_heads_total} attention heads ({prune_ratio*100:.0f}%)")

  head_mask = torch.ones_like(flat_importance)
  head_mask[indices_to_prune] = 0

  return head_mask.view_as(head_importance)

In [15]:
print("Computing attention head importance ... ")
head_importance = compute_head_importance(baseline_model, eval_loader)

head_mask = prune_attention_heads(baseline_model, head_importance, prune_ratio=0.3)
print(f"Remaining heads: {head_mask.sum().item():.0f} / {head_mask.numel()}")


Computing attention head importance ... 


Computing importance: 100%|██████████| 20/20 [00:07<00:00,  2.60it/s]

Pruning 81 out of 270 attention heads (30%)
Remaining heads: 189 / 270


# Knowledge Distillation

In [16]:
class TinyVLM(nn.Module):
  def __init__(self):
    super().__init__()

    self.vision_encoder = AutoModel.from_pretrained("google/vit-base-patch16-224")
    self.vision_hidden_size = 768

    self.language_model = AutoModel.from_pretrained("HuggingFaceTB/SmolLM-135M")
    self.language_hidden_size = 576

    self.projector = nn.Linear(self.vision_hidden_size, self.language_hidden_size)
    self.lm_head = nn.Linear(self.language_hidden_size, self.language_model.config.vocab_size, bias = False)

  def forward(self, pixel_values, input_ids, attention_mask = None, labels = None):
    vision_outputs = self.vision_encoder(pixel_values)
    image_features = vision_outputs.last_hidden_state
    image_embeds = self.projector(image_features)

    text_embeds = self.language_model.embed_tokens(input_ids)
    combined_embeds = torch.cat([image_embeds, text_embeds], dim = 1)

    batch_size = pixel_values.shape[0]
    image_attention = torch.ones(batch_size, image_embeds.shape[1], dtype=torch.long, device=pixel_values.device)

    if attention_mask is None:
      attention_mask = torch.ones(batch_size, image_embeds.shape[1],
                                  dtype = torch.long, device = pixel_values.device)
    combined_attention = torch.cat([image_attention, attention_mask], dim = 1)

    outputs = self.language_model(
        inputs_embeds = combined_embeds,
        attention_mask = combined_attention,
        output_hidden_states = True,
        return_dict = True
    )

    hidden_states = outputs.last_hidden_state
    logits = self.lm_head(hidden_states)

    loss = None
    if labels is not None:
      shift_logits = logits[:, image_embeds.shape[1]:-1, :].contiguous()
      shift_labels = labels[:, 1].contiguous()
      loss_fct = nn.CrossEntropyLoss(ignore_index=-100)
      loss = loss_fct(shift_logits.view(-1, shift_logits.size(-1)),
                      shift_labels.view(-1))
      return {"loss": loss, "logits": logits, "hidden_states": hidden_states}

  def count_parameters(self):
    return sum(p.numel() for p in self.parameters() if p.requires_grad)

  def get_model_size_mb(self):
    param_size = sum(p.numel() * p.element_size() for p in self.parameters())
    buffer_size = sum(b.numel() * b.element_size() for b in self.buffers())
    return (param_size + buffer_size) / 1024**2

In [17]:
student_model = TinyVLM().to(device)
print(f"\nStudent Model Statistics:")
print(f"  Total parameters: {student_model.count_parameters() / 1e6:.1f}M")
print(f"  Model size: {student_model.get_model_size_mb():.1f} MB")
print(f"  Size vs teacher: {baseline_model.get_model_size_mb() / student_model.get_model_size_mb():.2f}x smaller")


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] ViTModel LOAD REPORT from: google/vit-base-patch16-224
Key                 | Status     | 
--------------------+------------+-
classifier.bias     | UNEXPECTED | 
classifier.weight   | UNEXPECTED | 
pooler.dense.bias   | MISSING    | 
pooler.dense.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]


Student Model Statistics:
  Total parameters: 249.7M
  Model size: 695.8 MB
  Size vs teacher: 1.37x smaller


In [18]:
def distillation_loss(studen_logits, teacher_logits, labels, temperature=2.0, alpha=0.5):
  loss_fct = nn.CrossEntropyLoss(ignore_index=-100)
  hard_loss = loss_fct(student_logits.view(-1, student_logits.size(-1)), labels.view(-1))

  student_soft = F.log_softmax(student_logits / temperature, dim = -1)
  teacher_soft = F.softmax(teacher_logits / temperature, dim = -1)

  soft_loss = F.kl_div(student_soft, teacher_soft, reduction = 'batchmean') * (temperature ** 2)

  total_loss = alpha * hard_loss + (1 -alpha) * soft_loss

  return total_loss, hard_loss, soft_loss

In [19]:
def train_distilation(student, teacher, train_loader, num_epochs = 3, lr = 1e-4, temperature=2.0, alpha = 0.5):
  teacher.eval()
  student.train()

  optimizer = torch.optim.AdamW(student.parameters(), lr = lr)
  losses = []

  for epoch in range(num_epochs):
    epoch_loss = 0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch + 1}/{num_epochs}")

    for batch in pbar:
      pixel_values = batch["pixel_values"].to(device)
      input_ids = batch["input_ids"].to(device)
      attention_mask = batch["attention_mask"].to(device)
      labels = batch["labels"].to(device)

      with torch.no_grad():
        teacher_outputs = teacher(pixel_values, input_ids, attention_mask)
        teacher_logits = teacher_outputs["logits"]

      student_outputs = student(pixel_values, input_ids, attention_mask)
      student_logits = student_outputs["logits"]

      image_seq_len = teacher_outputs["hidden_states"].shape[1] - input_ids.shape[1]
      student_logits_aligned = student_logits[:, image_seq_len:-1, :]
      teacher_logits_aligned = teacher_logits[:, image_seq_len:-1, :]
      labels_aligned=labels[:, 1:]

      loss, hard_loss, soft_loss = distillation_loss(student_logits_aligned, teacher_logits_aligned, labels_aligned,
                                                     temperature = temperature, alpha=alpha)

      optimizer.zero_grad()
      loss.backward()
      optimizer.step()

      epoch_loss += loss.item()
      pbar.set_postfix({"loss": loss.item(), "hard": hard_loss.item(), "soft": soft_loss.item()})

    avg_loss = epoch_loss / len(train_loader)
    losses.append(avg_loss)
    print(f"Epoch {epoch + 1}/{num_epochs} - Average Loss: {avg_loss:.4f}")

  return student, losses


# Quantization + Distillation

In [20]:
student_quantized = torch.quantization.quantize_dynamic(
    copy.deepcopy(student_model),
    {nn.Linear},
    dtype=torch.qint8
)

/tmp/ipykernel_665/1303608331.py:1: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  student_quantized = torch.quantization.quantize_dynamic(


In [21]:
print(f"\nCombined Compression Results:")
print(f"  Original model: {baseline_model.get_model_size_mb():.1f} MB")
print(f"  Distilled student: {student_model.get_model_size_mb():.1f} MB")
print(f"  Quantized student: {student_quantized.get_model_size_mb():.1f} MB")
print(f"  Total compression: {baseline_model.get_model_size_mb() / student_quantized.get_model_size_mb():.2f}x")


Combined Compression Results:
  Original model: 952.4 MB
  Distilled student: 695.8 MB
  Quantized student: 57.0 MB
  Total compression: 16.69x
